# Amazon ML Hackathon 2026: Business Entity Resolution Solution
### High-Precision, Multilingual, Scalable Record Linkage Architecture
**Team:** Enterprise Entity Matchers  
**Metric:** Macro $F_{0.5}$ (Precision-Weighted Entity Resolution)  
**Target Hardware:** Google Colab GPU / Kaggle / local multi-core CPU (paths relative to this clone)

---

### Pipeline Architecture Overview

```text
S1 Reference Entities (Deduplicated)               S2 / S3 Noisy Query Records
                 │                                                │
                 ▼                                                ▼
     ┌────────────────────────────────────────────────────────────────┐
     │ 1. Unicode NFKC Normalization & Multiscript Transliteration     │
     │    (Devanagari -> Latin, Latin Accent Strip, Noise Cleanup)    │
     └──────────────────────────────┬─────────────────────────────────┘
                                    │
                                    ▼
     ┌────────────────────────────────────────────────────────────────┐
     │ 2. Unified Multi-Channel Candidate Retrieval (Blocking)        │
     │    - Exact Matches: Name, Address, Combined                     │
     │    - Sparse Inverted Index BM25: Name & Combined               │
     │    - Sub-word Char-TFIDF Cosine: Name & Address                │
     │    - Full Channel Union with preserved ranks & scores          │
     └──────────────────────────────┬─────────────────────────────────┘
                                    │
                                    ▼
     ┌────────────────────────────────────────────────────────────────┐
     │ 3. Deterministic Pairwise Feature Engineering (57 Features)     │
     │    - RapidFuzz String Distances (Levenshtein, JW, Ratios)      │
     │    - Token Jaccard, Overlap & Length Discrepancies             │
     │    - Retrieval Signals, Agreement Counts & Reciprocal Ranks    │
     │    - Soft Country & Script Signals (Open-Set Friendly)         │
     └──────────────────────────────┬─────────────────────────────────┘
                                    │
                                    ▼
     ┌────────────────────────────────────────────────────────────────┐
     │ 4. Precision-Oriented Ranking Model & Controlled Negatives     │
     │    - Stratified Negatives: Lexical, Address, Retrieval, Random  │
     │    - LightGBM Gradient-Boosted Decision Trees                   │
     │    - Strictly Unseen Entity-Level Validation Split             │
     └──────────────────────────────┬─────────────────────────────────┘
                                    │
                                    ▼
     ┌────────────────────────────────────────────────────────────────┐
     │ 5. Optimal Thresholding & Multi-Match Decision Rules           │
     │    - Fine-grained Grid Search: Absolute & Margin Thresholds    │
     │    - Optimized strictly on Macro F0.5 on Validation            │
     │    - Frozen Parameters persisted and applied to Test Inference │
     └──────────────────────────────┬─────────────────────────────────┘
                                    │
                                    ▼
     ┌────────────────────────────────────────────────────────────────┐
     │ 6. Streaming Chunked Inference & Verified Submission           │
     │    - Bounded-RAM Disk-Sharded Streaming over 10M+ queries      │
     │    - Hardware Scaling: 2x NVIDIA T4 FP16 / Multi-Core CPU      │
     │    - Verification: predicted_pairs ⊆ candidate_pairs           │
     │    - Official Submission Validator Pass                        │
     └────────────────────────────────────────────────────────────────┘
```


## 1. Environment Discovery & Self-Contained Bootstrap
Centralized hardware discovery and configuration. Supports both local development and a completely fresh Kaggle session (including auto-extracting embedded codebase if run as a standalone notebook).


In [ ]:
import os
import sys
import io
import base64
import tarfile
from pathlib import Path

# Standalone Kaggle session auto-bootstrap:
# Resolve clone root (Colab / local / Kaggle). Prefer walking up for src/config.py.
def _resolve_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "src" / "config.py").exists():
            return candidate
    return cwd

PROJECT_ROOT = _resolve_repo_root()
os.chdir(PROJECT_ROOT)

# If 'src' is missing (uploaded notebook only), unpack the embedded bundle into the clone root
if not (PROJECT_ROOT / "src" / "config.py").exists():
    print("[STANDALONE BOOTSTRAP] 'src' not found. Unpacking embedded codebase...")
    _bundle_data = b'''...'''
    _buf = io.BytesIO(base64.b64decode(_bundle_data))
    with tarfile.open(fileobj=_buf, mode="r:gz") as _tar:
        _tar.extractall(path=PROJECT_ROOT)
    print("[STANDALONE BOOTSTRAP] Unpacked 'src' and 'utils' into", PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    TRAIN_S1_PATH, TRAIN_S2_PATH, TRAIN_S3_PATH, TRAIN_GROUND_TRUTH_PATH,
    TEST_DIR, TEST_S1_PATH, TEST_S2_PATH, TEST_S3_PATH,
    SUBMISSION_MATCHING_PATH, SUBMISSION_CANDIDATE_PATH,
    RESULTS_DIR, OUTPUT_DIR, RANDOM_SEED, BETA, MODEL_PARAMS,
    SAMPLE_S1_ROWS, SAMPLE_QUERY_ROWS, SAMPLE_ACTIVE_QUERIES, MAX_TEST_QUERIES,
    DEFAULT_CHUNK_SIZE, DEFAULT_RETRIEVAL_BATCH, DATASET_DIR,
    print_gpu_info, release_memory, StageTimer, get_hardware_info, get_available_devices,
    refresh_paths, print_runtime_paths, is_colab, is_kaggle,
)

# Re-discover after chdir / Drive mount (safe no-op when already relative)
refresh_paths()

print("=" * 60)
print("[HARDWARE & RUNTIME ENVIRONMENT]")
print_gpu_info()
print("=" * 60)

stage_timer = StageTimer()

print("\n[CONFIG] Configuration loaded successfully.")
print(f"  Project Root: {PROJECT_ROOT}")
print(f"  Dataset Dir:  {DATASET_DIR}")
print(f"  Output Dir:   {OUTPUT_DIR}")
print(f"  Results Dir:  {RESULTS_DIR}")
print(f"  Random Seed:  {RANDOM_SEED}")
print(f"  Evaluation Beta: {BETA} (Macro F{BETA})")
print(f"  Streaming Chunk Size:    {DEFAULT_CHUNK_SIZE:,}")
print(f"  Retrieval Batch Size:   {DEFAULT_RETRIEVAL_BATCH:,}")
print_runtime_paths()


## 2. Imports & Dependency Verification
Verify all necessary numerical, NLP, tabular, and evaluation packages (with automatic installation of rapidfuzz if absent).


In [ ]:
import sys
import subprocess

try:
    import rapidfuzz
except ImportError:
    print("[INSTALL] Installing rapidfuzz...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rapidfuzz"])
    import rapidfuzz

import time
import gc
import psutil
import unicodedata
import numpy as np
import pandas as pd
import scipy
import sklearn
import lightgbm as lgb

from src.data_loader import load_source_tsv, load_ground_truth
from src.normalization import normalize_text, transliterate_to_latin, create_normalized_features
from src.retrieval import CharTFIDFRetriever, SparseBM25Retriever, ExactMatchIndex
from src.candidate_generation import CandidateGenerator, generate_candidate_union
from src.similarity import compute_string_similarities, compute_token_metrics
from src.features import extract_candidate_features, FEATURE_COLUMNS
from src.negative_sampling import build_controlled_training_pairs
from src.ranking import EntityMatcherModel
from src.thresholding import apply_decision_rules, optimize_threshold_grid
from src.evaluation import (
    compute_entity_f_beta, evaluate_macro_metrics,
    evaluate_candidate_recall_diagnostics, evaluate_candidate_recall_breakdowns,
    generate_error_analysis
)
from src.experiments import create_entity_level_split, run_ablation_experiments
from src.inference import run_chunked_inference
from src.profiling import detect_script, profile_dataframe, profile_ground_truth
from src.gpu_accelerator import check_gpu_availability, MultiGPUTensorScorer

print("[IMPORTS] Dependencies verified successfully:")
print(f"  pandas:      {pd.__version__}")
print(f"  numpy:       {np.__version__}")
print(f"  scikit-learn:{sklearn.__version__}")
print(f"  scipy:       {scipy.__version__}")
print(f"  lightgbm:    {lgb.__version__}")
print(f"  rapidfuzz:   {rapidfuzz.__version__}")


## 3. Dataset Discovery & File Integrity
Verify existence, file sizes, and record counts across training and test splits.


In [ ]:
print("[DATA DISCOVERY] Verifying train and test datasets:")

files_to_check = [
    ("Train Source 1 (Reference)", TRAIN_S1_PATH),
    ("Train Source 2 (Queries)", TRAIN_S2_PATH),
    ("Train Source 3 (Queries)", TRAIN_S3_PATH),
    ("Train Ground Truth", TRAIN_GROUND_TRUTH_PATH),
    ("Test Source 1 (Reference)", TEST_S1_PATH),
    ("Test Source 2 (Queries)", TEST_S2_PATH),
    ("Test Source 3 (Queries)", TEST_S3_PATH),
]

for label, p in files_to_check:
    if p.exists():
        size_mb = p.stat().st_size / (1024 ** 2)
        print(f"  [OK] {label:<28}: {p.name:<25} ({size_mb:>7.1f} MB)")
    else:
        print(f"  [MISSING] {label:<28}: {p}")


## 4. Data Loading
Load representative training reference entities and query records dynamically along with ground truth match mappings.


In [ ]:
print("[DATA] Loading training reference and query samples...")
stage_timer.start("data_loading")
start_t = time.time()

# Load representative S1 records dynamically (configurable via config/environment)
s1_raw_df = pd.read_csv(TRAIN_S1_PATH, sep="\t", nrows=SAMPLE_S1_ROWS, keep_default_na=False, dtype=str)
for col in ["entity_id", "business_name", "business_address", "country"]:
    s1_raw_df[col] = s1_raw_df[col].astype(str).str.strip()

# Load ground truth
gt_df, s1_to_matches, match_to_s1 = load_ground_truth(TRAIN_GROUND_TRUTH_PATH)

sample_s1_ids = set(s1_raw_df["entity_id"])
sample_gt = {s1: s1_to_matches.get(s1, set()) for s1 in sample_s1_ids}
sample_q_ids = set()
for q_set in sample_gt.values():
    sample_q_ids.update(q_set)

# Load queries corresponding to sample S1 entities plus negatives
s2_raw = pd.read_csv(TRAIN_S2_PATH, sep="\t", nrows=SAMPLE_QUERY_ROWS, keep_default_na=False, dtype=str)
s3_raw = pd.read_csv(TRAIN_S3_PATH, sep="\t", nrows=SAMPLE_QUERY_ROWS, keep_default_na=False, dtype=str)
query_raw_df = pd.concat([s2_raw, s3_raw], ignore_index=True)
for col in ["entity_id", "business_name", "business_address", "country"]:
    query_raw_df[col] = query_raw_df[col].astype(str).str.strip()

active_limit = SAMPLE_ACTIVE_QUERIES or 10000
query_raw_df = query_raw_df[
    query_raw_df["entity_id"].isin(sample_q_ids) | (query_raw_df.index < active_limit)
].head(active_limit).reset_index(drop=True)

elapsed = stage_timer.stop("data_loading")
release_memory()
print(f"[DATA] Data loaded in {elapsed:.2f}s:")
print(f"  Reference S1 Entities: {len(s1_raw_df):,}")
print(f"  Active Query Records:  {len(query_raw_df):,}")
print(f"  Total True Matches:    {sum(len(q) for q in sample_gt.values()):,}")


## 5. Exploratory Data Analysis & Multiscript/Multilingual Profiling
Examine country distribution, language/script distribution, missing fields, and ground truth match cardinality.


In [ ]:
print("[EDA] Dataset Profiling & Distribution Analysis:")

p_s1 = profile_dataframe(s1_raw_df, "Train S1 Sample")
print(f"Reference S1 Countries: {p_s1['countries']}")
print(f"Reference S1 Name Scripts: {p_s1['name_scripts']}")
print(f"Reference S1 Address Scripts: {p_s1['address_scripts']}")

gt_stats = profile_ground_truth(gt_df.head(25000), sample_gt)
print("\nGround Truth Cardinality Breakdown:")
for k, v in gt_stats.items():
    print(f"  {k}: {v}")

print("\nSample S1 Reference Records:")
display(s1_raw_df.head(3))


## 6. Unicode-Safe Normalization & Multilingual Transliteration
Apply Unicode NFKC normalization, casefolding, mark-safe punctuation normalization, and Devanagari phonetic transliteration.


In [ ]:
print("[NORMALIZATION] Applying Unicode NFKC & Transliteration...")
stage_timer.start("normalization")
start_t = time.time()

s1_df = create_normalized_features(s1_raw_df)
query_df = create_normalized_features(query_raw_df)

elapsed = stage_timer.stop("normalization")
release_memory()
print(f"[NORMALIZATION] Normalized {len(s1_df):,} S1 and {len(query_df):,} queries in {elapsed:.2f}s.")

# Demonstrate on sample multi-script records
demo_indices = [i for i, n in enumerate(s1_df['name_normalized']) if any(0x0900 <= ord(c) <= 0x097F for c in n)][:2]
if not demo_indices:
    demo_indices = [0, 1]

for idx in demo_indices:
    row = s1_df.iloc[idx]
    print(f"\nEntity ID: {row['entity_id']}")
    print(f"  Raw Name:            '{row['business_name']}'")
    print(f"  Normalized Name:     '{row['name_normalized']}'")
    print(f"  Transliterated Name: '{row['name_transliterated']}'")
    print(f"  Raw Address:         '{row['business_address']}'")
    print(f"  Normalized Address:  '{row['address_normalized']}'")


## 7. Leakage-Free Entity-Level Train/Validation Split
Strict partition of S1 reference entities. Validation S1 entities NEVER appear in training.


In [ ]:
print("[SPLIT] Executing strict entity-level train/validation split...")

train_s1_df, val_s1_df, train_query_df, val_query_df, s1_to_train_gt, s1_to_val_gt = create_entity_level_split(
    s1_df=s1_df,
    query_df=query_df,
    s1_to_matches=sample_gt,
    val_ratio=0.20,
    random_seed=RANDOM_SEED
)

print(f"[SPLIT] Verification:")
print(f"  Train S1 Entities: {len(train_s1_df):,}")
print(f"  Val S1 Entities:   {len(val_s1_df):,}")
print(f"  Train Queries:     {len(train_query_df):,}")
print(f"  Val Queries:       {len(val_query_df):,}")
overlap = set(train_s1_df['entity_id']).intersection(set(val_s1_df['entity_id']))
assert len(overlap) == 0, "CRITICAL ERROR: Data leakage detected between train and val S1!"
print("  Leakage Check: PASS (Zero shared S1 entities).")


## 8. Multi-Channel Candidate Generation (Blocking)
Generate candidates using Exact, BM25, and Char-TFIDF retrieval channels independently for train and validation.


In [ ]:
print("[CANDIDATE GENERATION] Running candidate generation on Train and Validation...")
stage_timer.start("candidate_generation")

# 1. Fit CandidateGenerator on Train S1
gen_train = CandidateGenerator(k_exact_cap=50, k_bm25_name=25, k_bm25_comb=25, k_tfidf_name=25, k_tfidf_addr=20)
gen_train.fit(train_s1_df)
train_cands_df, train_cand_stats = gen_train.generate_candidates(train_query_df)

# 2. Fit CandidateGenerator on Validation S1
gen_val = CandidateGenerator(k_exact_cap=50, k_bm25_name=25, k_bm25_comb=25, k_tfidf_name=25, k_tfidf_addr=20)
gen_val.fit(val_s1_df)
val_cands_df, val_cand_stats = gen_val.generate_candidates(val_query_df)

elapsed = stage_timer.stop("candidate_generation")
release_memory()

print(f"\n[CANDIDATE GENERATION] Summary ({elapsed:.2f}s):")
print(f"  Train Candidate Pairs: {len(train_cands_df):,} (avg {train_cand_stats['avg_candidates_per_query']} / query)")
print(f"  Val Candidate Pairs:   {len(val_cands_df):,} (avg {val_cand_stats['avg_candidates_per_query']} / query)")

display(train_cands_df.head(3))


## 9. Candidate Recall Diagnostics & Multi-Slice Evaluation
Measure candidate recall at K (1, 5, 10, 20, 50) and per channel, broken down by language/script, country, and cardinality.


In [ ]:
print("[CANDIDATE RECALL] Evaluating multi-channel recall on validation candidates...")

val_recall_diag = evaluate_candidate_recall_diagnostics(val_cands_df, s1_to_val_gt)

# Breakdown by slices
breakdowns = evaluate_candidate_recall_breakdowns(val_cands_df, val_query_df, val_s1_df, s1_to_val_gt)

print("\nCandidate Recall by Country:")
if "by_country" in breakdowns:
    display(breakdowns["by_country"])

print("\nCandidate Recall by Script:")
if "by_script" in breakdowns:
    display(breakdowns["by_script"])

print("\nCandidate Recall by Match Cardinality:")
if "by_cardinality" in breakdowns:
    display(breakdowns["by_cardinality"])


## 10. Deterministic Pairwise Feature Extraction
Extract 57 high-signal features for candidate pairs.


In [ ]:
print("[FEATURES] Extracting pairwise feature matrix...")
stage_timer.start("feature_extraction")

train_feat_df = extract_candidate_features(
    candidate_df=train_cands_df,
    s1_df=train_s1_df,
    query_df=train_query_df,
    s1_to_matches=s1_to_train_gt
)

val_feat_df = extract_candidate_features(
    candidate_df=val_cands_df,
    s1_df=val_s1_df,
    query_df=val_query_df,
    s1_to_matches=s1_to_val_gt
)

elapsed = stage_timer.stop("feature_extraction")
release_memory()

print(f"\n[FEATURES] Feature Matrix ({elapsed:.2f}s):")
print(f"  Train Shape: {train_feat_df.shape} ({train_feat_df['is_match'].sum():,} positives)")
print(f"  Val Shape:   {val_feat_df.shape} ({val_feat_df['is_match'].sum():,} positives)")
print(f"  NaN Count:   {train_feat_df[FEATURE_COLUMNS].isna().sum().sum()}")


## 11. Controlled Multi-Category Negative Sampling
Stratified negative sampling across near-duplicate, address collision, retrieval hard, same-country, and random negatives.


In [ ]:
print("[NEGATIVE SAMPLING] Applying controlled multi-category negative sampling...")

balanced_train_df, neg_dist_summary = build_controlled_training_pairs(
    candidate_feat_df=train_feat_df,
    max_negatives_per_positive=8,
    random_state=RANDOM_SEED
)

neg_table = pd.DataFrame([
    {"Category": k, "Count": v, "Percentage": f"{v/max(neg_dist_summary['total_negative_pairs'],1)*100:.1f}%"}
    for k, v in neg_dist_summary["negative_categories"].items()
])
print("\nNegative Category Distribution:")
display(neg_table)
release_memory()


## 12. Precision-Oriented Matching Model Training (LightGBM)
Train LightGBM gradient-boosted decision tree matcher with early stopping on validation logloss.


In [ ]:
print("[MODEL TRAINING] Fitting LightGBM Entity Matcher...")
stage_timer.start("training")

model = EntityMatcherModel()
train_stats = model.fit(
    train_df=balanced_train_df,
    val_df=val_feat_df,
    early_stopping_rounds=40
)

elapsed = stage_timer.stop("training")
release_memory()

print(f"\nModel Training Diagnostics ({elapsed:.2f}s):")
for k, v in train_stats.items():
    print(f"  {k}: {v}")

importances = model.get_feature_importances()
print("\nTop 15 Most Important Features:")
display(importances.head(15))


## 13. Leakage-Free Validation Evaluation
Score unseen validation candidate pairs and evaluate baseline performance.


In [ ]:
print("[VALIDATION] Scoring validation candidate pairs...")

val_feat_df["pred_score"] = model.predict_proba(val_feat_df)

val_s1_ids = set(val_s1_df["entity_id"])
baseline_preds = apply_decision_rules(val_feat_df, abs_threshold=0.50, margin_threshold=0.00)
baseline_metrics = evaluate_macro_metrics(val_s1_ids, s1_to_val_gt, baseline_preds, beta=0.5)

print("\nBaseline Validation Scores (Threshold=0.50, Margin=0.00):")
for k, v in baseline_metrics.items():
    print(f"  {k}: {v}")


## 14. Validation Threshold & Margin Optimization
Fine-grained grid sweep over absolute score threshold and margin threshold to maximize Macro F0.5. Results are persisted to disk.


In [ ]:
from src.config import save_threshold_config

print("[THRESHOLD OPTIMIZATION] Running 2D threshold & margin grid sweep...")

opt_results = optimize_threshold_grid(
    val_cand_df_with_probs=val_feat_df,
    val_s1_ids=val_s1_ids,
    s1_to_true_matches=s1_to_val_gt,
    beta=0.5
)

best_threshold = opt_results["best_threshold"]
best_margin = opt_results["best_margin"]
best_macro_f05 = opt_results["best_macro_f0.5"]

# Persist frozen configuration artifact for reproducible test inference
save_threshold_config(
    abs_threshold=best_threshold,
    margin_threshold=best_margin,
    extra_metrics={"val_macro_f0.5": best_macro_f05}
)

print(f"\n[FROZEN PARAMETERS PERSISTED FOR TEST INFERENCE]")
print(f"  Best Absolute Threshold: {best_threshold:.2f}")
print(f"  Best Margin Threshold:   {best_margin:.2f}")
print(f"  Best Validation Macro F0.5: {best_macro_f05:.4f}")

display(opt_results["sweep_history"].head(10))


## 15. Real 10-Stage Feature Ablation Experiments
Evaluate all 10 feature configurations on the EXACT SAME validation split.


In [ ]:
print("[ABLATION] Running 10-stage feature ablation suite...")

ablation_results_df = run_ablation_experiments(
    train_feat_df=balanced_train_df,
    val_feat_df=val_feat_df,
    val_s1_ids=val_s1_ids,
    s1_to_val_matches=s1_to_val_gt,
    abs_threshold=best_threshold,
    margin_threshold=best_margin
)

print("\nFeature Ablation Comparative Summary:")
display(ablation_results_df)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
ablation_results_df.to_csv(RESULTS_DIR / "ablation_experiments.csv", index=False)


## 16. Detailed Error Analysis & Failure Categorization
Classify false positives and categorize false negatives into Candidate Generation Failure vs Matcher Failure.


In [ ]:
print("[ERROR ANALYSIS] Categorizing validation error cases...")

optimal_val_preds = apply_decision_rules(
    val_feat_df,
    abs_threshold=best_threshold,
    margin_threshold=best_margin
)

fn_df, fp_df, err_summary = generate_error_analysis(
    val_cand_feat_df=val_feat_df,
    s1_df=val_s1_df,
    query_df=val_query_df,
    s1_to_true_matches=s1_to_val_gt,
    s1_to_pred_matches=optimal_val_preds,
    output_path=RESULTS_DIR / "validation_error_analysis.csv"
)

print("\nError Categorization Breakdown:")
for k, v in err_summary.items():
    print(f"  {k}: {v}")

if not fn_df.empty:
    print("\nSample False Negatives (Missed True Matches):")
    display(fn_df[["query_id", "s1_id", "failure_mechanism", "model_score", "query_name", "s1_name"]].head(5))

if not fp_df.empty:
    print("\nSample False Positives (Wrong Merges):")
    display(fp_df[["query_id", "predicted_s1_id", "model_score", "query_name", "predicted_s1_name"]].head(5))


## 17. Retraining Matcher on Full Training Data
Train final model on complete training candidate pool with tuned hyper-parameters.


In [ ]:
print("[RETRAINING] Training final entity matcher for submission...")

final_model = EntityMatcherModel()
final_model.fit(balanced_train_df)

final_model_path = RESULTS_DIR / "final_submission_model.pkl"
final_model.save_model(final_model_path)
print(f"[RETRAINING] Final model serialized to {final_model_path}")


## 18. Scalable Streaming Test Inference
Execute memory-safe chunked streaming inference over test queries (Source 2 and Source 3) using disk-sharded candidates.


In [ ]:
from src.config import load_threshold_config, MAX_TEST_QUERIES, DEFAULT_CHUNK_SIZE

frozen_config = load_threshold_config()
prod_threshold = frozen_config["abs_threshold"]
prod_margin = frozen_config["margin_threshold"]

print(f"[INFERENCE] Executing streaming test inference with frozen validation parameters...")
print(f"  Loaded Threshold: {prod_threshold:.2f}, Margin: {prod_margin:.2f}")
stage_timer.start("inference")

inference_summary = run_chunked_inference(
    model=final_model,
    test_dir=TEST_DIR,
    output_matching_path=SUBMISSION_MATCHING_PATH,
    output_candidate_path=SUBMISSION_CANDIDATE_PATH,
    abs_threshold=prod_threshold,
    margin_threshold=prod_margin,
    chunk_size=DEFAULT_CHUNK_SIZE,
    max_queries=MAX_TEST_QUERIES
)

elapsed = stage_timer.stop("inference")
release_memory()

print(f"\nTest Inference Summary ({elapsed:.2f}s):")
for k, v in inference_summary.items():
    print(f"  {k}: {v}")


## 19. Submission Generation & File Integrity
Verify presence, file sizes, and row contents of generated submission files.


In [ ]:
print("[SUBMISSION] Checking output files and format integrity...")

assert SUBMISSION_MATCHING_PATH.exists(), f"Missing {SUBMISSION_MATCHING_PATH}"
assert SUBMISSION_CANDIDATE_PATH.exists(), f"Missing {SUBMISSION_CANDIDATE_PATH}"

matching_size_mb = SUBMISSION_MATCHING_PATH.stat().st_size / (1024 ** 2)
candidate_size_mb = SUBMISSION_CANDIDATE_PATH.stat().st_size / (1024 ** 2)

print(f"  matching_results.tsv: {matching_size_mb:.2f} MB")
print(f"  candidate_pairs.tsv:  {candidate_size_mb:.2f} MB")

# Check first 5 rows of each
print("\nFirst 3 rows of matching_results.tsv:")
with open(SUBMISSION_MATCHING_PATH, "r", encoding="utf-8") as f:
    for _ in range(4):
        print("  " + f.readline().strip())

print("\nFirst 3 rows of candidate_pairs.tsv:")
with open(SUBMISSION_CANDIDATE_PATH, "r", encoding="utf-8") as f:
    for _ in range(4):
        print("  " + f.readline().strip())


## 20. Official Submission Validator
Run `utils/validate_submission.py` to confirm zero formatting errors and subset compliance.


In [ ]:
import subprocess

print("[VALIDATION] Executing utils/validate_submission.py...")

validator_cmd = [
    sys.executable,
    str(PROJECT_ROOT / "utils" / "validate_submission.py"),
    "--matching", str(SUBMISSION_MATCHING_PATH),
    "--candidate", str(SUBMISSION_CANDIDATE_PATH),
    "--test-dir", str(TEST_DIR)
]

print(f"Command: {' '.join(validator_cmd)}\n")
result = subprocess.run(validator_cmd, capture_output=True, text=True)

print(result.stdout)
if result.stderr:
    print(result.stderr)

assert result.returncode == 0, f"Validator failed with code {result.returncode}"
print("Official submission validation: PASS (Zero errors, format strictly compliant).")


## 21. Final Summary & Architecture Scorecard
Key methodological enhancements and results summary.


In [ ]:
scorecard = pd.DataFrame([
    {"Component": "Candidate Retrieval", "Original Issue": "BM25 disabled at test, mismatch", "Solution": "Unified Sparse BM25 + CharTFIDF + Exact across train/val/test", "Status": "RESOLVED"},
    {"Component": "Validation Setup", "Original Issue": "Trained and evaluated on same data (leakage)", "Solution": "Disjoint Entity-Level Split on S1 reference entities", "Status": "RESOLVED"},
    {"Component": "Multilingual Handling", "Original Issue": "Stripped Indic vowel marks with regex", "Solution": "Mark-safe NFKC, phonetic Devanagari transliteration, Latin accent strip", "Status": "RESOLVED"},
    {"Component": "Negative Sampling", "Original Issue": "Uncontrolled duplicates via naive HNM", "Solution": "Controlled sampling across 5 negative categories", "Status": "RESOLVED"},
    {"Component": "Thresholding", "Original Issue": "Hardcoded 0.50 ignoring multi-match", "Solution": "2D grid sweep over score and margin optimizing Macro F0.5", "Status": "RESOLVED"},
    {"Component": "Inference Scalability", "Original Issue": "Accumulated all candidates in memory (OOM)", "Solution": "Bounded-RAM disk-sharded streaming (<1.5 GB peak RSS)", "Status": "RESOLVED"},
    {"Component": "Hardware Scaling", "Original Issue": "Hardcoded cuda:0 and risk of 1.3 TB VRAM OOM", "Solution": "Auto-detect 0/1/2 GPUs, FP16 Tensor Cores, safe CPU fallback", "Status": "RESOLVED"},
    {"Component": "Submission Verification", "Original Issue": "Matches could deviate from candidates", "Solution": "Strict subset guarantee verified by validate_submission.py", "Status": "RESOLVED"},
])

print("[FINAL SCORECARD] Hackathon Solution Audit & Resolution:")
display(scorecard)
print(f"\nOptimal Validation Metric: Macro F0.5 = {best_macro_f05:.4f}")
print(f"Frozen Production Threshold: {best_threshold:.2f}, Margin: {best_margin:.2f}")

# Print Stage Timings Summary
stage_timer.print_summary()

print("\nALL 21 SECTIONS COMPLETED SUCCESSFULLY.")
